# RBC Subtype Classifier (ViT baseline)

This notebook trains a **Vision Transformer (ViT-S/16)** baseline to classify red blood cell (RBC) subtypes from **grayscale, cropped** microscopy images organized as **one folder per class**.

**Highlights**
- Uses `torchvision.datasets.ImageFolder` with **grayscale → 3-channel** conversion (so ViT can use pretrained weights).
- Strong, safe augmentations (RandAugment, flips, slight affine).
- Class imbalance handled via **weighted sampling**.
- **Head-first finetuning** (freeze backbone), then **unfreeze top blocks**.
- Metrics: accuracy, macro-F1, per-class F1, confusion matrix.
- Saves best model and a one-line **inference script**.

> If your environment is offline (no internet), set `PRETRAINED = False` in the Model cell. Otherwise leave it `True` to download weights once.


In [ ]:
# --- repo-root anchor (added during reorganisation) ---
# Walks up from the working directory to the repo root, so every path below
# resolves whether this notebook is run from code/notebooks/ or the repo root.
from pathlib import Path as _P
REPO_ROOT = _P.cwd().resolve()
while not (REPO_ROOT / 'requirements.txt').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR       = REPO_ROOT / 'data'
MODELS_DIR     = REPO_ROOT / 'models'
RESULTS_DIR    = REPO_ROOT / 'results'
REFERENCE_DIR  = REPO_ROOT / 'reference'
VALIDATION_DIR = RESULTS_DIR / 'validation'
# Staging area for freshly trained weights. Training ALWAYS writes here,
# never straight into models/, so released checkpoints are never overwritten.
CKPT_DIR       = MODELS_DIR / 'rbc_ckpts'
CKPT_DIR.mkdir(parents=True, exist_ok=True)
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)
print('repo root:', REPO_ROOT)


In [8]:
# ===== 1) Config =====
import os
from pathlib import Path

# CHANGE THIS to the folder that contains your 7 class subfolders.
# Example folder layout:
# data_root/
#   A_Discocyte/
#   B_Cup-shape/
#   C_Stomatocyte/
#   D_Reticulocyte/
#   E_Echinocyte/
#   F_Granular/
#   G_ISC/
DATA_ROOT = DATA_DIR / 'Alldataset_for_subtypeclassification'

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 4

# Training schedule
EPOCHS_HEAD = 10         # head-only finetuning
EPOCHS_FULL = 40         # unfreeze top blocks + fine-tune
BASE_LR_HEAD = 3e-4
BASE_LR_FULL = 1e-4
WEIGHT_DECAY = 0.05

# Other
SEED = 42
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15
PRETRAINED = True        # set to False if offline
SAVE_DIR = CKPT_DIR
SAVE_DIR.mkdir(parents=True, exist_ok=True)

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)


In [9]:
# ===== 2) Imports and seeding =====
import random
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler
from torchvision import datasets, transforms
from torchvision.transforms import InterpolationMode
import timm

import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import itertools

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cpu')

In [18]:
# ===== 3) Dataset & stratified split =====
assert DATA_ROOT.exists(), f"Data folder not found: {DATA_ROOT.resolve()}"

# We will load in RGB then convert to grayscale->3ch to keep ViT happy.
# (ImageFolder loads PIL images; we enforce grayscale and replicate to 3 channels.)
base_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE), interpolation=InterpolationMode.BICUBIC),
])

full_dataset = datasets.ImageFolder(root=str(DATA_ROOT), transform=base_transform)
num_classes = len(full_dataset.classes)
class_names = full_dataset.classes
print('Classes:', class_names)

# Build stratified indices for train/val/test
targets = [y for _, y in full_dataset.samples]

from collections import defaultdict
indices_by_class = defaultdict(list)
for idx, (_, y) in enumerate(full_dataset.samples):
    indices_by_class[y].append(idx)

train_idx, val_idx, test_idx = [], [], []
for y, idx_list in indices_by_class.items():
    n = len(idx_list)
    n_val = int(round(n * VAL_SPLIT))
    n_test = int(round(n * TEST_SPLIT))
    n_train = n - n_val - n_test
    random.shuffle(idx_list)
    train_idx += idx_list[:n_train]
    val_idx   += idx_list[n_train:n_train+n_val]
    test_idx  += idx_list[n_train+n_val:]

# ===== Debug subset (per-class downsample) =====
DEBUG_SMALL = True  # set to True to enable
K_TRAIN, K_VAL, K_TEST = 100, 30, 30   # images per class; adjust as you like

if DEBUG_SMALL:
    def take_per_class(original_idx, k):
        # keep at most k samples per class from the provided index list
        chosen = []
        # map sample index -> class
        idx2cls = {i: full_dataset.samples[i][1] for i in original_idx}
        # group by class
        from collections import defaultdict
        by_cls = defaultdict(list)
        for i in original_idx:
            by_cls[idx2cls[i]].append(i)
        # take first k per class
        for cls, lst in by_cls.items():
            random.shuffle(lst)
            chosen += lst[:min(k, len(lst))]
        return chosen

    train_idx = take_per_class(train_idx, K_TRAIN)
    val_idx   = take_per_class(val_idx,   K_VAL)
    test_idx  = take_per_class(test_idx,  K_TEST)

    print("DEBUG subset sizes:",
          len(train_idx), len(val_idx), len(test_idx))


len(train_idx), len(val_idx), len(test_idx)


Classes: ['A_Discocyte', 'B_Cup-shape', 'C_Stomatocyte', 'D_Reticulocyte', 'E_Echinocyte', 'F_Granular', 'G_ISC']
DEBUG subset sizes: 700 210 210


(700, 210, 210)

In [19]:
# ===== 4) Transforms & DataLoaders =====
# base (only used for listing classes earlier; can keep or remove)
base_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE), interpolation=InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0), interpolation=InterpolationMode.BICUBIC),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomAffine(degrees=10, translate=(0.02, 0.02), scale=(0.98, 1.02)),
    transforms.RandAugment(num_ops=2, magnitude=7),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_test_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE), interpolation=InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

full_dataset_train = datasets.ImageFolder(root=str(DATA_ROOT), transform=train_transform)
full_dataset_val   = datasets.ImageFolder(root=str(DATA_ROOT), transform=val_test_transform)
full_dataset_test  = datasets.ImageFolder(root=str(DATA_ROOT), transform=val_test_transform)

train_ds = Subset(full_dataset_train, train_idx)
val_ds   = Subset(full_dataset_val,   val_idx)
test_ds  = Subset(full_dataset_test,  test_idx)

# Weighted sampling to fight class imbalance
train_targets = [full_dataset.samples[i][1] for i in train_idx]
class_counts = np.bincount(train_targets, minlength=num_classes)
class_weights = 1.0 / np.maximum(class_counts, 1)
sample_weights = [class_weights[t] for t in train_targets]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

len(train_loader), len(val_loader), len(test_loader)


(22, 7, 7)

In [20]:
# ===== 5) Model: ViT-S/16 (DINO/DeiT/ViT) via timm =====
# We pick a small ViT for fast finetuning. You can swap to other timm models if you prefer.
# Good small choices: 'vit_small_patch16_224.dino', 'vit_small_patch16_224.augreg_in21k', 'deit_small_patch16_224'
model_name = 'vit_small_patch16_224.dino'

model = timm.create_model(model_name, pretrained=PRETRAINED, num_classes=num_classes)
# DropPath/LayerScale etc. are already configured in the checkpoint; timm handles it nicely.

# We'll "head-first" finetune: freeze the backbone (all but classifier head)
def freeze_backbone(m):
    for name, p in m.named_parameters():
        if 'head' not in name and 'fc' not in name and 'classifier' not in name:
            p.requires_grad = False

freeze_backbone(model)
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                              lr=BASE_LR_HEAD, weight_decay=WEIGHT_DECAY)
lr_schedule = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_HEAD + EPOCHS_FULL)
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

print(f"Model: {model_name}, params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")


Model: vit_small_patch16_224.dino, params: 21.7M


/var/folders/08/4mg4trr52cz6gmp1vkmspz080000gn/T/ipykernel_34533/1022909938.py:22: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())


In [21]:
# ===== 6) Training helpers =====
def train_one_epoch(model, loader, optimizer, scaler):
    model.train()
    total, correct, loss_sum = 0, 0, 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            logits = model(x)
            loss = criterion(logits, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        loss_sum += loss.item() * x.size(0)
        preds = logits.argmax(1)
        correct += (preds == y).sum().item()
        total += x.size(0)
    return loss_sum/total, correct/total

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total, correct, loss_sum = 0, 0, 0.0
    all_preds, all_targets = [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)
        loss_sum += loss.item() * x.size(0)
        preds = logits.argmax(1)
        correct += (preds == y).sum().item()
        total += x.size(0)
        all_preds.append(preds.cpu())
        all_targets.append(y.cpu())
    all_preds = torch.cat(all_preds).numpy()
    all_targets = torch.cat(all_targets).numpy()
    return loss_sum/total, correct/total, all_preds, all_targets

def plot_confusion_matrix(cm, classes, normalize=False, title='Confusion matrix'):
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, None]
    plt.figure(figsize=(6, 6))
    plt.imshow(cm, interpolation='nearest')
    plt.title(title)
    plt.colorbar()
    tick_marks = range(len(classes))
    plt.xticks(tick_marks, classes, rotation=45, ha='right')
    plt.yticks(tick_marks, classes)
    fmt = '.2f' if normalize else 'd'
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt),
                 horizontalalignment='center',
                 verticalalignment='center')
    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.show()


In [22]:
# ===== 7) Train: head-only, then partial unfreeze =====
best_val_acc = 0.0
best_path = SAVE_DIR / 'best_vit.pth'

# Phase 1: head-only
for epoch in range(1, EPOCHS_HEAD+1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, scaler)
    val_loss, val_acc, _, _ = evaluate(model, val_loader)
    lr_schedule.step()
    print(f"[Head {epoch:02d}/{EPOCHS_HEAD}] train_acc={tr_acc:.3f} val_acc={val_acc:.3f}")
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({'model': model.state_dict(), 'classes': class_names}, best_path)

# Phase 2: unfreeze top blocks for full finetuning
# For ViT in timm, encoder blocks are at model.blocks; we unfreeze last N blocks.
N_UNFREEZE = 4
for i in range(len(model.blocks)-N_UNFREEZE, len(model.blocks)):
    for p in model.blocks[i].parameters():
        p.requires_grad = True

optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                              lr=BASE_LR_FULL, weight_decay=WEIGHT_DECAY)

for epoch in range(1, EPOCHS_FULL+1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, scaler)
    val_loss, val_acc, _, _ = evaluate(model, val_loader)
    lr_schedule.step()
    print(f"[Full {epoch:02d}/{EPOCHS_FULL}] train_acc={tr_acc:.3f} val_acc={val_acc:.3f}")
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({'model': model.state_dict(), 'classes': class_names}, best_path)

print(f"Best val acc: {best_val_acc:.3f}. Saved to {best_path}")


/var/folders/08/4mg4trr52cz6gmp1vkmspz080000gn/T/ipykernel_34533/2575743790.py:8: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):


[Head 01/10] train_acc=0.137 val_acc=0.195
[Head 02/10] train_acc=0.193 val_acc=0.143
[Head 03/10] train_acc=0.174 val_acc=0.205
[Head 04/10] train_acc=0.263 val_acc=0.238
[Head 05/10] train_acc=0.240 val_acc=0.210
[Head 06/10] train_acc=0.310 val_acc=0.271
[Head 07/10] train_acc=0.290 val_acc=0.262
[Head 08/10] train_acc=0.299 val_acc=0.300
[Head 09/10] train_acc=0.350 val_acc=0.310
[Head 10/10] train_acc=0.336 val_acc=0.190
[Full 01/40] train_acc=0.406 val_acc=0.357
[Full 02/40] train_acc=0.419 val_acc=0.348
[Full 03/40] train_acc=0.457 val_acc=0.386
[Full 04/40] train_acc=0.486 val_acc=0.552
[Full 05/40] train_acc=0.497 val_acc=0.505
[Full 06/40] train_acc=0.527 val_acc=0.510
[Full 07/40] train_acc=0.514 val_acc=0.519
[Full 08/40] train_acc=0.580 val_acc=0.524


: 

In [ ]:
# ===== 8) Test & report =====
# Load best checkpoint
ckpt = torch.load(best_path, map_location='cpu')
model.load_state_dict(ckpt['model'])

test_loss, test_acc, y_pred, y_true = evaluate(model, test_loader)
print(f"Test acc: {test_acc:.3f}")

print("\nClassification report (per-class F1):\n")
print(classification_report(y_true, y_pred, target_names=class_names, digits=3))

cm = confusion_matrix(y_true, y_pred)
plot_confusion_matrix(cm, class_names, normalize=False, title='Confusion Matrix (counts)')
plot_confusion_matrix(cm, class_names, normalize=True, title='Confusion Matrix (normalized)')


In [ ]:
# ===== 9) Inference util =====
# Simple function to run a single image (path) through the trained model.
from PIL import Image

inference_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE), interpolation=InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

@torch.no_grad()
def predict_image(img_path: str):
    model.eval()
    img = Image.open(img_path).convert('L')  # grayscale
    x = inference_transform(img)
    x = x.unsqueeze(0).to(device)
    logits = model(x)
    probs = torch.softmax(logits, dim=1).cpu().numpy().squeeze()
    idx = int(np.argmax(probs))
    return class_names[idx], float(probs[idx]), {class_names[i]: float(probs[i]) for i in range(len(class_names))}

# Example:
# pred, conf, all_probs = predict_image('path/to/one/cell.png')
# pred, conf
